# Hospital Readmission Prediction (10-Year Diabetes Dataset)**Group 14 -- Machine Learning (Waterloo Data Science Certificate)**Team: Jamie Bennion, Mohammad Bhatti, Meriem Meghari, Elaheh Nahvi, Jemar Ugale## ObjectiveUsing 10 years (1999-2008) of clinical data from 130 US hospitals (~100K inpatient encounters,50+ features), predict whether a diabetic patient will be readmitted to hospital within 30 daysof discharge.Dataset source: [10 Years Diabetes Dataset on Kaggle](https://www.kaggle.com/datasets/jimschacko/10-years-diabetes-dataset)(`diabetes.csv` -- not included in this repo due to size; see README for download instructions).

## Setup

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')from sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import train_test_split, cross_val_score, GridSearchCVfrom sklearn.metrics import (    classification_report, confusion_matrix, accuracy_score, f1_score,    roc_curve, roc_auc_score)from sklearn.linear_model import LogisticRegressionfrom sklearn.tree import DecisionTreeClassifierfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysisfrom sklearn.naive_bayes import GaussianNBfrom sklearn.ensemble import (    AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier)from sklearn.utils import resamplefrom category_encoders import BinaryEncoderfrom tabulate import tabulateseed = 123

# 1. Exploratory Data Analysis & Preprocessing

### Reading the data

In [ ]:
diabetes = pd.read_csv('diabetes.csv')diabetes.head()

Drop the columns that only identify a patient/encounter and carry no predictive signal, plus `weight` (per [Strack et al., 2014](https://www.hindawi.com/journals/bmri/2014/781670/), this field wasn't consistently captured) and `payer_code` (not a meaningful predictor).

In [ ]:
data = diabetes.copy()data.drop(['id', 'encounter_id', 'patient_nbr', 'weight', 'payer_code'], axis=1, inplace=True)data.info()

### Handling `?` placeholder values

In [ ]:
for col in data.columns:    n_missing = (data[col] == '?').sum()    if n_missing > 0:        print(col, n_missing)

`race` and `medical_specialty` get an explicit `Other` category; the three diagnosis codes get a placeholder numeric code since the count of missing values is small relative to the dataset.

In [ ]:
data['race'].replace('?', 'Other', inplace=True)data['medical_specialty'].replace('?', 'Other', inplace=True)data['diag_1'].replace('?', '1000', inplace=True)data['diag_2'].replace('?', '1000', inplace=True)data['diag_3'].replace('?', '1000', inplace=True)

### Target variable: 30-day readmission

The raw `readmitted` field has three values (`NO`, `>30`, `<30`). We collapse this into a binary target: readmitted within 30 days (1) vs. not (0).

In [ ]:
data['readmitted'] = data['readmitted'].map({'NO': 0, '>30': 0, '<30': 1})data = data.rename(columns={'readmitted': 'readmitted_within_30_days'})data['readmitted_within_30_days'].value_counts()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))labels = ['0', '1']sns.countplot(x=data.readmitted_within_30_days, data=data, palette='pastel', ax=ax[0], edgecolor='.3')data.readmitted_within_30_days.value_counts().plot.pie(    autopct='%1.2f%%', ax=ax[1], colors=['#66a3ff', '#facc99'],    labels=labels, explode=(0, 0.05), startangle=120,    textprops={'fontsize': 12, 'color': '#0a0a00'})plt.show()

**88.84% of patients were not readmitted within 30 days** -- a meaningfully imbalanced target, which we account for before training (see Section 2).

### Numerical features: correlation & outliers

In [ ]:
num_features = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications',                 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']def heatmap(df):    correlations = df.corr()    cmap = sns.diverging_palette(220, 10, as_cmap=True)    fig, ax = plt.subplots(figsize=(10, 10))    sns.heatmap(correlations, cmap=cmap, vmax=1.0, center=0, fmt='.2f', square=True,                linewidths=.5, annot=True, cbar_kws={'shrink': .75})    plt.tight_layout()    plt.show()heatmap(data[num_features])

No strongly correlated numerical features. `time_in_hospital` correlates moderately with `num_medications` and `num_procedures`, which makes sense -- longer stays mean more treatment.

In [ ]:
def boxplots(df, columns):    fig, ax = plt.subplots(nrows=2, ncols=4, figsize=(16, 7))    for i, col in enumerate(columns):        sns.boxplot(x=df[col], palette='pastel', ax=ax[i // 4][i % 4])    plt.tight_layout()    plt.show()boxplots(data[num_features], num_features)

### Categorical features

#### Race -- one-hot encoded (no natural ordering, low cardinality)

In [ ]:
data.groupby(['race']).size().sort_values(ascending=False)

In [ ]:
data_encoded = pd.get_dummies(data, columns=['race'], prefix=['enc'])

#### Gender -- drop the small `Unknown/Invalid` group, binary encode

In [ ]:
data_encoded['gender'].replace('Unknown/Invalid', np.nan, inplace=True)data_encoded.dropna(subset=['gender'], inplace=True)data_encoded['gender'].replace({'Male': 0, 'Female': 1}, inplace=True)

#### Age -- ordinal encode to preserve the natural ordering of the bands

In [ ]:
age_ranges = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',              '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']age_dict = dict(zip(age_ranges, range(len(age_ranges))))data_encoded['age_encoded'] = data_encoded['age'].map(age_dict)data_encoded.drop(['age'], axis=1, inplace=True)

#### Admission type, discharge disposition, admission source

These are high-cardinality categorical IDs. We group rare/overlapping categories (per the Kaggle data dictionary) and binary-encode to control dimensionality, instead of one-hot encoding dozens of sparse columns.

In [ ]:
# group NULL/Not Mapped/Not Available admission types (5, 6, 8) into one categorydata_encoded['admission_type_id'] = data_encoded['admission_type_id'].replace([5, 6, 8], 5)data_encoded = BinaryEncoder(cols=['admission_type_id']).fit_transform(data_encoded)

Discharge dispositions 11, 13, 14, 19, 20, 21 all correspond to the patient expiring or being discharged to hospice -- these encounters can't be "readmitted" in any meaningful sense, so we drop them before encoding.

In [ ]:
hospice_death_codes = [11, 13, 14, 19, 20, 21]data_encoded = data_encoded[~data_encoded['discharge_disposition_id'].isin(hospice_death_codes)]data_encoded = data_encoded.reset_index(drop=True)data_encoded = BinaryEncoder(cols=['discharge_disposition_id']).fit_transform(data_encoded)

In [ ]:
# group unknown/not-mapped/not-available admission sources into one categorydata_encoded['admission_source_id'] = data_encoded['admission_source_id'].replace([9, 15, 17, 20, 21], 17)data_encoded = BinaryEncoder(cols=['admission_source_id']).fit_transform(data_encoded)

#### Lab/medication result fields -- one-hot or binary as appropriate

In [ ]:
data_encoded = pd.get_dummies(data_encoded, columns=['max_glu_serum'], prefix=['enc'])data_encoded = pd.get_dummies(data_encoded, columns=['A1Cresult'], prefix=['enc'])data_encoded['change'].replace({'No': 0, 'Ch': 1}, inplace=True)data_encoded['diabetesMed'].replace({'No': 0, 'Yes': 1}, inplace=True)

#### Drug columns -- collapse to a simple "was the patient on this medication" flag

In [ ]:
drug_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',             'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone',             'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide',             'citoglipton', 'insulin', 'glyburide.metformin', 'glipizide.metformin',             'glimepiride.pioglitazone', 'metformin.rosiglitazone', 'metformin.pioglitazone']for col in drug_cols:    data_encoded[col] = data_encoded[col].replace({'No': 0, 'Steady': 1, 'Up': 1, 'Down': 1})

`medical_specialty`, `diag_1`, `diag_2`, and `diag_3` are extremely high-cardinality (80-950+ distinct values) and would blow up dimensionality if encoded directly -- we drop them rather than introduce hundreds of sparse columns.

In [ ]:
data_encoded.drop(['diag_1', 'diag_2', 'diag_3', 'medical_specialty'], axis=1, inplace=True)data_encoded.shape

# 2. Modeling## Train/test split & class imbalance

In [ ]:
y = data_encoded['readmitted_within_30_days']X = data_encoded.drop(columns='readmitted_within_30_days')X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)print(X_train.shape, X_test.shape)

With an 88/12 class split, a model can score well on raw accuracy while barely predicting the minority (readmitted) class at all. We balance the **training set only** by undersampling the majority class down to the minority count -- the test set is left untouched so evaluation still reflects real-world class proportions.

In [ ]:
# undersampling reconstructed from the project report -- original notebook's resampling step wasn't savedtrain_df = X_train.copy()train_df['readmitted_within_30_days'] = y_train.valuesmajority = train_df[train_df['readmitted_within_30_days'] == 0]minority = train_df[train_df['readmitted_within_30_days'] == 1]majority_downsampled = resample(    majority, replace=False, n_samples=len(minority), random_state=42)train_balanced = pd.concat([majority_downsampled, minority])y_train = train_balanced['readmitted_within_30_days']X_train = train_balanced.drop(columns='readmitted_within_30_days')X_train = StandardScaler().fit_transform(X_train)X_test = StandardScaler().fit_transform(X_test)print('Balanced training set:', y_train.value_counts().to_dict())

## Baseline model comparison

Run a quick comparison across several classifiers with default hyperparameters to see which families of models are worth tuning further.

In [ ]:
def based_models():    return [        ('LR', LogisticRegression(max_iter=1000)),        ('LDA', LinearDiscriminantAnalysis()),        ('KNN', KNeighborsClassifier()),        ('RF', RandomForestClassifier(n_estimators=150, random_state=seed)),        ('NB', GaussianNB()),        ('AB', AdaBoostClassifier()),        ('GBM', GradientBoostingClassifier()),        ('ET', ExtraTreesClassifier()),    ]def baseline(X_train, y_train, X_test, y_test, models):    rows = []    for name, model in models:        model.fit(X_train, y_train)        cv_results = cross_val_score(model, X_train, y_train, scoring='accuracy')        test_score = model.score(X_test, y_test)        rows.append([name, cv_results.mean(), test_score])    print(tabulate(rows, headers=['Model', 'CV Accuracy', 'Test Accuracy'], tablefmt='orgtbl'))baseline(X_train, y_train, X_test, y_test, based_models())

## Hyperparameter tuning: Random Forest

In [ ]:
rf_params = {    'max_depth': [2, 5, 8],    'n_estimators': [100, 200, 500, 700],    'max_features': [3, 5, 8],    'min_samples_split': [2, 5, 10],}rf_grid = GridSearchCV(    RandomForestClassifier(random_state=seed), rf_params, cv=3, n_jobs=-1, verbose=1).fit(X_train, y_train)rf_grid.best_params_

## Final evaluation

In [ ]:
best_rf = rf_grid.best_estimator_y_pred = best_rf.predict(X_test)print('Test accuracy:', round(accuracy_score(y_test, y_pred) * 100, 2), '%')print(classification_report(y_test, y_pred))

In [ ]:
plt.title('Confusion Matrix -- Random Forest (tuned)')sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, cmap='YlGn', fmt='d')plt.xlabel('Predicted')plt.ylabel('Actual')plt.show()

In [ ]:
rf_probs = best_rf.predict_proba(X_test)[:, 1]rf_fpr, rf_tpr, rf_thresholds = roc_curve(y_test, rf_probs)rf_auc = roc_auc_score(y_test, rf_probs)print('ROC AUC:', round(rf_auc, 2))plt.plot(rf_fpr, rf_tpr, label='ROC Curve')plt.xlabel('False Positive Rate')plt.ylabel('True Positive Rate (Recall)')plt.legend(loc=4)plt.show()

In [ ]:
print('F1 score (weighted):', round(f1_score(y_test, y_pred, average='weighted'), 2))

## Results & Conclusion**Tuned Random Forest: AUC = 0.61, weighted-average F1 = 0.63.** Readmission is a genuinely hardprediction problem -- an AUC of 0.61 is modestly better than chance, which is consistent with thebroader literature on this dataset rather than a sign of a modeling mistake. F1 here is thesupport-weighted average across both classes (via `classification_report`), which is the standardway to summarize performance on an imbalanced target like this one.The team's official project report compared Logistic Regression, Random Forest, and GradientBoosting after hyperparameter tuning and selected **Gradient Boosting as the team's final pick**(~58.4% test accuracy, F1 = 0.614) -- close to, though not identical to, the tuned Random Forestresult reproduced in this notebook. All three models performed similarly after tuning, which thereport notes as well.**Limitations & future work (from the report):** the dataset's imbalance is a fundamental challenge;the team noted that sourcing a more balanced dataset, or combining features more deliberately duringfeature engineering instead of dropping low-correlation ones outright, could improve results further.

## Appendix: Feature ReferenceFeature descriptions sourced from the [Kaggle dataset documentation](https://www.kaggle.com/datasets/jimschacko/10-years-diabetes-dataset)and [Strack et al., 2014](https://www.hindawi.com/journals/bmri/2014/781670/).| Feature | Description ||---|---|| `race` | Caucasian, Asian, African American, Hispanic, Other || `gender` | Male, Female, Unknown/Invalid || `age` | Grouped in 10-year intervals || `admission_type_id` | 9 distinct values -- e.g. emergency, urgent, elective, newborn || `discharge_disposition_id` | 29 distinct values -- e.g. discharged to home, expired || `admission_source_id` | 21 distinct values -- e.g. physician referral, ER, transfer || `time_in_hospital` | Days between admission and discharge || `medical_specialty` | Specialty of the admitting physician (84 distinct values) || `num_lab_procedures` / `num_procedures` / `num_medications` | Counts during the encounter || `number_outpatient` / `number_emergency` / `number_inpatient` | Visits in the year preceding the encounter || `diag_1`, `diag_2`, `diag_3` | Primary/secondary diagnoses (ICD-9 codes) || `max_glu_serum` | Glucose serum test result range, or not measured || `A1Cresult` | A1C test result range, or not measured || `change` | Whether diabetic medication dosage/type changed || `diabetesMed` | Whether any diabetic medication was prescribed || 23 medication columns | Per-drug dosage status: up / down / steady / no || `readmitted` | `<30`, `>30`, or `NO` -- collapsed to a binary target in this notebook |